# Notebook 07: Cross-Metro Comparison & Topological Analysis

**Goal**: Test the land-only tax shift hypothesis in other densely
populated U.S. metro areas, and use Topological Data Analysis (TDA)
to quantify structural differences in urban development patterns
across metros.

**Fiscal comparison**: For metros with open land/building assessed
value data, run the same revenue-neutral simulation.

**TDA addition**: Use persistent homology (H0, H1) on property
lat/lon point clouds to create a topological fingerprint of each
city's development pattern — fragmentation, voids, connectivity —
and relate these to the tax shift outcomes.

**Target metros**:
1. Cook County, IL (our primary analysis)
2. Philadelphia, PA — OpenDataPhilly
3. Washington, D.C. — DC Open Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import json
from pathlib import Path
from io import StringIO

sns.set_theme(style="whitegrid")
Path("../data/external").mkdir(parents=True, exist_ok=True)

In [ ]:
# Install TDA libraries if needed
# !pip install ripser persim

from ripser import ripser
from persim import plot_diagrams
import persim

---
## PART A: Cross-Metro Fiscal Comparison
---

### 1. Load Cook County Results

In [ ]:
cook = pd.read_parquet("../data/processed/tax_simulation_results.parquet")
cook_homeowners = cook[cook["is_homeowner"]].copy()

# Load full assessment for lat/lon
cook_assess = pd.read_parquet("../data/processed/assessment_decomposed.parquet")

print(f"Cook County homeowners: {len(cook_homeowners):,}")
print(f"Cook County winner % (Scenario B): {100*cook_homeowners['winner_B'].mean():.1f}%")

### 2. Load Philadelphia Data

In [ ]:
def load_philadelphia():
    """Load Philadelphia property data from OpenDataPhilly API."""
    filepath = "../data/external/philadelphia_properties.parquet"
    
    if Path(filepath).exists():
        print("Loading cached Philadelphia data...")
        return pd.read_parquet(filepath)
    
    print("Downloading Philadelphia property data...")
    # lat/lng extracted from the_geom via PostGIS; land/bldg split from taxable columns
    query = """
    SELECT parcel_number, market_value, sale_price, sale_date,
           category_code, total_livable_area, total_area,
           number_of_bedrooms, number_of_bathrooms, year_built,
           taxable_land, taxable_building,
           exempt_land, exempt_building,
           location,
           ST_Y(the_geom) AS lat,
           ST_X(the_geom) AS lng
    FROM opa_properties_public
    WHERE category_code LIKE '1%'
    AND market_value > 0
    LIMIT 300000
    """
    url = "https://phl.carto.com/api/v2/sql"
    try:
        resp = requests.get(url, params={"q": query, "format": "csv"}, timeout=120)
        resp.raise_for_status()
        df = pd.read_csv(StringIO(resp.text))
        df.to_parquet(filepath, index=False)
        print(f"Saved {len(df):,} Philadelphia properties.")
        return df
    except Exception as e:
        print(f"Philadelphia download failed: {e}")
        print("Will use placeholder for fiscal comparison.")
        return None

philly = load_philadelphia()
if philly is not None:
    print(f"Philadelphia: {philly.shape}")
    print(f"Columns: {philly.columns.tolist()[:20]}")


### 3. Load Washington, D.C. Data

In [ ]:
def load_dc():
    """Load DC property data from DC Open Data (ArcGIS export)."""
    filepath = "../data/external/dc_properties.parquet"

    if Path(filepath).exists():
        print("Loading cached DC data...")
        return pd.read_parquet(filepath)

    print("Downloading DC property data...")
    # Direct CSV export from ArcGIS Hub (includes NEWLAND, NEWIMPR, NEWTOTAL)
    url = (
        "https://opendata.arcgis.com/api/v3/datasets/"
        "dc698a3e39ff48059777ef8de1bbed8a_0/downloads/data"
        "?format=csv&spatialRefId=4326&where=1=1"
    )
    try:
        resp = requests.get(url, timeout=300)
        resp.raise_for_status()
        # utf-8-sig strips the BOM character present in this export
        df = pd.read_csv(StringIO(resp.content.decode("utf-8-sig")), low_memory=False)
        df.columns = df.columns.str.strip()
        # Cast mixed-type columns to string so pyarrow can serialise them
        for col in df.columns:
            if df[col].dtype == object:
                df[col] = df[col].astype(str)

        # --- Fetch coordinates from Tax Lot Points layer ---
        # Layer 22 of Property_and_Land_WebMercator has SSL + point geometry
        print("Fetching DC parcel coordinates...")
        coords_url = (
            "https://maps2.dcgis.dc.gov/dcgis/rest/services/DCGIS_DATA/"
            "Property_and_Land_WebMercator/MapServer/22/query"
        )
        all_coords = []
        offset = 0
        page_size = 2000
        while True:
            params = {
                "where": "1=1",
                "outFields": "SSL",
                "returnGeometry": "true",
                "resultOffset": offset,
                "resultRecordCount": page_size,
                "f": "geojson",
            }
            r = requests.get(coords_url, params=params, timeout=60)
            feats = r.json().get("features", [])
            if not feats:
                break
            for feat in feats:
                coords = feat.get("geometry", {}).get("coordinates", [None, None])
                ssl = feat.get("properties", {}).get("SSL")
                all_coords.append({"SSL": ssl, "lon": coords[0], "lat": coords[1]})
            offset += page_size
            if len(feats) < page_size:
                break

        coords_df = pd.DataFrame(all_coords)
        # SSL in the tax lot layer has spaces; strip to match the attribute table
        coords_df["SSL"] = coords_df["SSL"].str.strip()
        df["SSL"] = df["SSL"].astype(str).str.strip()
        df = df.merge(coords_df, on="SSL", how="left")
        print(f"Coordinates joined: {df['lat'].notna().sum():,} of {len(df):,} parcels matched")

        df.to_parquet(filepath, index=False)
        print(f"Saved {len(df):,} DC properties.")
        return df
    except Exception as e:
        print(f"DC download failed: {e}")
        print("Will use placeholder for fiscal comparison.")
        return None

dc = load_dc()
if dc is not None:
    print(f"DC: {dc.shape}")
    print(f"Columns: {dc.columns.tolist()[:20]}")


### 4. Standardize & Run Tax Simulation

In [ ]:
def find_column(df, keywords, exclude=None):
    """Find a column name containing any of the keywords."""
    exclude = exclude or []
    for col in df.columns:
        col_lower = col.lower()
        if any(kw in col_lower for kw in keywords):
            if not any(ex in col_lower for ex in exclude):
                return col
    return None

def standardize_metro(df, metro_name, land_col, bldg_col, 
                       total_col=None, lat_col=None, lon_col=None):
    """Standardize metro data for tax simulation and TDA."""
    result = df.copy()
    result["metro"] = metro_name
    
    result["land_value"] = pd.to_numeric(result[land_col], errors="coerce")
    result["bldg_value"] = pd.to_numeric(result[bldg_col], errors="coerce")
    
    if total_col and total_col in result.columns:
        result["total_value"] = pd.to_numeric(result[total_col], errors="coerce")
    else:
        result["total_value"] = result["land_value"] + result["bldg_value"]
    
    # Coordinates
    if lat_col and lat_col in result.columns:
        result["lat"] = pd.to_numeric(result[lat_col], errors="coerce")
    if lon_col and lon_col in result.columns:
        result["lon"] = pd.to_numeric(result[lon_col], errors="coerce")
    
    result = result[
        (result["total_value"] > 0) &
        (result["land_value"] >= 0) &
        (result["bldg_value"] >= 0)
    ].copy()
    
    result["site_ratio"] = result["land_value"] / result["total_value"]
    return result

def run_tax_simulation(df, metro_name):
    """Run revenue-neutral land-only tax shift simulation."""
    total_base = df["total_value"].sum()
    land_base = df["land_value"].sum()
    
    df["current_tax_share"] = df["total_value"] / total_base
    
    if land_base > 0:
        df["proposed_tax_share"] = df["land_value"] / land_base
    else:
        df["proposed_tax_share"] = df["current_tax_share"]
    
    df["winner"] = df["proposed_tax_share"] < df["current_tax_share"]
    
    pct_winners = 100 * df["winner"].mean()
    winners = df[df["winner"]]
    losers = df[~df["winner"]]
    
    # Approximate dollar change (normalize to $5000 avg bill)
    avg_bill = 5000
    df["change"] = (df["proposed_tax_share"] - df["current_tax_share"]) * avg_bill * len(df)
    
    return {
        "metro": metro_name,
        "n_properties": len(df),
        "pct_winners": pct_winners,
        "median_site_ratio": df["site_ratio"].median(),
        "mean_site_ratio": df["site_ratio"].mean(),
        "std_site_ratio": df["site_ratio"].std(),
        "data": df,
    }

In [ ]:
# === Process each metro ===
metro_results = []

# Cook County
cook_result = {
    "metro": "Cook County, IL",
    "n_properties": len(cook_homeowners),
    "pct_winners": 100 * cook_homeowners["winner_B"].mean(),
    "median_site_ratio": cook_homeowners["site_value_ratio"].median(),
    "mean_site_ratio": cook_homeowners["site_value_ratio"].mean(),
    "std_site_ratio": cook_homeowners["site_value_ratio"].std(),
    "data": cook_homeowners.rename(columns={
        "site_value_ratio": "site_ratio",
        "loc_latitude": "lat", "loc_longitude": "lon",
        "winner_B": "winner",
    }),
}
metro_results.append(cook_result)

# Philadelphia
if philly is not None:
    land_col = find_column(philly, ["land"], exclude=["total"])
    bldg_col = find_column(philly, ["building", "bldg", "improv"], exclude=["total"])
    total_col = find_column(philly, ["market_value", "total_value"])
    lat_col = find_column(philly, ["lat"])
    lon_col = find_column(philly, ["lng", "lon", "long"])

    print(f"\nPhiladelphia columns found:")
    print(f"  Land: {land_col}, Bldg: {bldg_col}, Total: {total_col}")
    print(f"  Lat: {lat_col}, Lon: {lon_col}")

    if land_col and bldg_col:
        philly_std = standardize_metro(philly, "Philadelphia, PA",
                                        land_col, bldg_col, total_col,
                                        lat_col, lon_col)
        philly_result = run_tax_simulation(philly_std, "Philadelphia, PA")
        metro_results.append(philly_result)
        print(f"  Processed: {philly_result['n_properties']:,} properties, "
              f"{philly_result['pct_winners']:.1f}% winners")
    else:
        print("  Could not identify land/building columns.")

# DC
if dc is not None:
    # Use exact column names — find_column substring match picks up LANDAREA for "land"
    land_col = "NEWLAND" if "NEWLAND" in dc.columns else find_column(dc, ["newland"])
    bldg_col = "NEWIMPR" if "NEWIMPR" in dc.columns else find_column(dc, ["newimpr", "improv"])
    total_col = "NEWTOTAL" if "NEWTOTAL" in dc.columns else find_column(dc, ["newtotal", "total"], exclude=["land", "improv"])
    lat_col = "lat" if "lat" in dc.columns else None
    lon_col = "lon" if "lon" in dc.columns else None

    print(f"\nDC columns found:")
    print(f"  Land: {land_col}, Bldg: {bldg_col}, Total: {total_col}")
    print(f"  Lat: {lat_col}, Lon: {lon_col}")

    if land_col and bldg_col:
        dc_std = standardize_metro(dc, "Washington, D.C.",
                                    land_col, bldg_col, total_col,
                                    lat_col, lon_col)
        dc_result = run_tax_simulation(dc_std, "Washington, D.C.")
        metro_results.append(dc_result)
        print(f"  Processed: {dc_result['n_properties']:,} properties, "
              f"{dc_result['pct_winners']:.1f}% winners")
    else:
        print("  Could not identify land/building columns.")

print(f"\nTotal metros processed: {len(metro_results)}")


### 5. Fiscal Comparison Visualization

In [ ]:
comparison = pd.DataFrame([
    {k: v for k, v in r.items() if k != "data"} for r in metro_results
])

print("=" * 80)
print("  CROSS-METRO FISCAL COMPARISON")
print("=" * 80)
print(comparison.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# % Winners
ax = axes[0]
bars = ax.bar(comparison["metro"], comparison["pct_winners"],
              color="steelblue", edgecolor="black")
ax.axhline(50, color="red", linestyle="--", label="50% threshold")
ax.set_title("% Homeowners Paying Less\nUnder Land-Only Tax")
ax.set_ylabel("% Paying Less")
ax.legend()
ax.tick_params(axis="x", rotation=20)
for bar, val in zip(bars, comparison["pct_winners"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f"{val:.1f}%", ha="center", fontsize=10, fontweight="bold")

# Median site ratio
ax = axes[1]
bars = ax.bar(comparison["metro"], comparison["median_site_ratio"] * 100,
              color="teal", edgecolor="black")
ax.set_title("Median Land/Total Value Ratio")
ax.set_ylabel("Site Ratio (%)")
ax.tick_params(axis="x", rotation=20)
for bar, val in zip(bars, comparison["median_site_ratio"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f"{val:.1%}", ha="center", fontsize=10)

# Site ratio distributions
ax = axes[2]
colors = ["steelblue", "coral", "green", "purple"]
for i, r in enumerate(metro_results):
    data = r["data"]["site_ratio"].dropna().clip(0, 1)
    ax.hist(data, bins=80, alpha=0.4, label=r["metro"],
            color=colors[i % len(colors)], density=True)
ax.set_title("Site Ratio Distributions")
ax.set_xlabel("Site Value / Total Value")
ax.set_ylabel("Density")
ax.legend()

plt.suptitle("Cross-Metro Comparison: Revenue-Neutral Land-Only Tax Shift",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../outputs/figures/07_fiscal_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

---
## PART B: Topological Data Analysis (Persistent Homology)
---

We use persistent homology to create a *topological fingerprint* of
each metro's urban development pattern. By computing the Vietoris-Rips
filtration on property lat/lon point clouds, we extract two types of
topological features:

- **H0 (connected components)**: Tracks how development clusters and
  fragments. High persistence in H0 = isolated pockets of development
  with gaps — the kind of land speculation pattern a site-value tax
  is meant to discourage.

- **H1 (loops/holes)**: Captures "rings" of development surrounding
  undeveloped cores. High H1 persistence = donut-shaped patterns
  where valuable central land is held vacant.

### 6. Prepare Point Clouds

In [ ]:
def prepare_point_cloud(df, lat_col="lat", lon_col="lon", 
                         sample_size=3000, seed=42):
    """
    Sample and normalize property coordinates for TDA.
    
    We normalize lat/lon to a common scale (approximate km) so that
    persistence values are comparable across metros at different
    geographic scales.
    """
    coords = df[[lat_col, lon_col]].dropna()
    coords = coords[(coords[lat_col] != 0) & (coords[lon_col] != 0)]
    
    if len(coords) < 100:
        print(f"  WARNING: Only {len(coords)} valid coordinates.")
        return None
    
    # Sample
    n = min(sample_size, len(coords))
    sampled = coords.sample(n, random_state=seed).values
    
    # Convert degrees to approximate km
    # 1 degree latitude \u2248 111 km
    # 1 degree longitude \u2248 111 \u00d7 cos(latitude) km
    center_lat = np.radians(sampled[:, 0].mean())
    sampled_km = np.column_stack([
        sampled[:, 0] * 111.0,                          # lat \u2192 km
        sampled[:, 1] * 111.0 * np.cos(center_lat),     # lon \u2192 km
    ])
    
    # Center at origin for numerical stability
    sampled_km -= sampled_km.mean(axis=0)
    
    return sampled_km

# Build point clouds for each metro
metro_point_clouds = {}

for result in metro_results:
    metro = result["metro"]
    data = result["data"]
    
    # Find coordinate columns
    lat_col = "lat" if "lat" in data.columns else None
    lon_col = "lon" if "lon" in data.columns else None
    
    if lat_col is None:
        for c in data.columns:
            if "lat" in c.lower():
                lat_col = c
                break
    if lon_col is None:
        for c in data.columns:
            if "lon" in c.lower() or "lng" in c.lower():
                lon_col = c
                break
    
    if lat_col and lon_col:
        print(f"Preparing point cloud for {metro}...")
        cloud = prepare_point_cloud(data, lat_col, lon_col, sample_size=3000)
        if cloud is not None:
            metro_point_clouds[metro] = cloud
            print(f"  {len(cloud)} points, "
                  f"extent: {cloud[:, 0].max() - cloud[:, 0].min():.1f} \u00d7 {cloud[:, 1].max() - cloud[:, 1].min():.1f} km")
    else:
        print(f"No coordinates available for {metro}")

print(f"\nPoint clouds ready for {len(metro_point_clouds)} metros.")

### 7. Compute Persistent Homology

In [ ]:
def compute_topo_features(coords, maxdim=1):
    """
    Compute persistent homology and extract summary statistics.
    
    Returns a dict of topological features and the raw persistence
    diagrams for visualization.
    """
    # Compute Vietoris-Rips persistence
    result = ripser(coords, maxdim=maxdim)
    diagrams = result["dgms"]
    
    features = {}
    for dim in range(maxdim + 1):
        dgm = diagrams[dim]
        # Remove infinite-persistence features (the one component that
        # persists forever in H0)
        finite = dgm[np.isfinite(dgm[:, 1])]
        lifetimes = finite[:, 1] - finite[:, 0]
        
        if len(lifetimes) > 0:
            features[f"H{dim}_count"] = len(lifetimes)
            features[f"H{dim}_mean_persistence"] = lifetimes.mean()
            features[f"H{dim}_max_persistence"] = lifetimes.max()
            features[f"H{dim}_std_persistence"] = lifetimes.std()
            features[f"H{dim}_total_persistence"] = lifetimes.sum()
            
            # Persistence entropy (normalized)
            # High entropy = many features with similar persistence
            # Low entropy = a few dominant features
            if lifetimes.sum() > 0:
                probs = lifetimes / lifetimes.sum()
                probs = probs[probs > 0]
                features[f"H{dim}_entropy"] = -np.sum(probs * np.log2(probs))
            else:
                features[f"H{dim}_entropy"] = 0
        else:
            for stat in ["count", "mean_persistence", "max_persistence",
                         "std_persistence", "total_persistence", "entropy"]:
                features[f"H{dim}_{stat}"] = 0
    
    return features, diagrams

# Compute for each metro
topo_results = {}
topo_diagrams = {}

for metro, coords in metro_point_clouds.items():
    print(f"Computing persistent homology for {metro}...")
    features, diagrams = compute_topo_features(coords, maxdim=1)
    topo_results[metro] = features
    topo_diagrams[metro] = diagrams
    
    print(f"  H0: {features['H0_count']} features, "
          f"max persistence = {features['H0_max_persistence']:.3f} km")
    print(f"  H1: {features['H1_count']} features, "
          f"max persistence = {features['H1_max_persistence']:.3f} km")

topo_df = pd.DataFrame(topo_results).T
topo_df.index.name = "metro"
print("\n=== Topological Feature Summary ===")
print(topo_df.to_string())

### 8. Visualize Persistence Diagrams

In [ ]:
n_metros = len(topo_diagrams)
fig, axes = plt.subplots(2, max(n_metros, 2), 
                          figsize=(6 * max(n_metros, 2), 10))

if n_metros == 1:
    axes = axes.reshape(2, 1)

for i, (metro, diagrams) in enumerate(topo_diagrams.items()):
    # Row 1: Persistence diagrams
    ax = axes[0, i]
    plot_diagrams(diagrams, ax=ax, show=False)
    ax.set_title(f"{metro}\nPersistence Diagram")
    
    # Row 2: Point cloud with color by density
    ax = axes[1, i]
    coords = metro_point_clouds[metro]
    ax.scatter(coords[:, 1], coords[:, 0], s=1, alpha=0.3, color="steelblue")
    ax.set_xlabel("East-West (km)")
    ax.set_ylabel("North-South (km)")
    ax.set_title(f"{metro}\nProperty Point Cloud")
    ax.set_aspect("equal")

# Hide unused axes
for j in range(n_metros, axes.shape[1]):
    axes[0, j].set_visible(False)
    axes[1, j].set_visible(False)

plt.suptitle("Persistent Homology of Urban Development Patterns",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../outputs/figures/07_persistence_diagrams.png", dpi=150, bbox_inches="tight")
plt.show()

### 9. Compare Topological Features Across Metros

In [ ]:
# Select key features for comparison
key_features = [
    "H0_count", "H0_max_persistence", "H0_mean_persistence", "H0_entropy",
    "H1_count", "H1_max_persistence", "H1_mean_persistence", "H1_entropy",
]
available_features = [f for f in key_features if f in topo_df.columns]

fig, axes = plt.subplots(2, 4, figsize=(20, 10))

feature_labels = {
    "H0_count": "H0: # Components",
    "H0_max_persistence": "H0: Max Persistence (km)",
    "H0_mean_persistence": "H0: Mean Persistence (km)",
    "H0_entropy": "H0: Persistence Entropy",
    "H1_count": "H1: # Loops",
    "H1_max_persistence": "H1: Max Persistence (km)",
    "H1_mean_persistence": "H1: Mean Persistence (km)",
    "H1_entropy": "H1: Persistence Entropy",
}

colors_h0 = "steelblue"
colors_h1 = "coral"

for ax, feat in zip(axes.flat, available_features):
    color = colors_h0 if feat.startswith("H0") else colors_h1
    vals = topo_df[feat]
    bars = ax.bar(vals.index, vals.values, color=color, edgecolor="black", alpha=0.8)
    ax.set_title(feature_labels.get(feat, feat), fontsize=10)
    ax.tick_params(axis="x", rotation=30, labelsize=8)
    for bar, val in zip(bars, vals.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f"{val:.3f}" if val < 10 else f"{val:.0f}",
                ha="center", va="bottom", fontsize=8)

# Hide unused axes
for ax in axes.flat[len(available_features):]:
    ax.set_visible(False)

plt.suptitle("Topological Fingerprints: Cross-Metro Comparison",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../outputs/figures/07_topo_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

### 10. Relate Topology to Tax Shift Outcomes

In [ ]:
# Merge topological features with fiscal results
combined = comparison.set_index("metro").join(topo_df)

if len(combined) >= 2 and "H0_max_persistence" in combined.columns:
    print("=== Topology \u2194 Tax Shift Relationship ===\n")
    print(combined[["pct_winners", "median_site_ratio", 
                     "H0_count", "H0_max_persistence", "H0_entropy",
                     "H1_count", "H1_max_persistence"]].to_string())
    
    # Scatter: H0 max persistence vs % winners
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    ax = axes[0]
    ax.scatter(combined["H0_max_persistence"], combined["pct_winners"],
               s=200, color="steelblue", edgecolor="black", zorder=5)
    for idx, row in combined.iterrows():
        ax.annotate(idx, (row["H0_max_persistence"], row["pct_winners"]),
                    textcoords="offset points", xytext=(10, 5), fontsize=9)
    ax.set_xlabel("H0 Max Persistence (km)\n(Development Fragmentation)")
    ax.set_ylabel("% Homeowners Paying Less")
    ax.set_title("Fragmentation vs Tax Shift Benefit")
    ax.axhline(50, color="red", linestyle="--", alpha=0.5)
    
    ax = axes[1]
    ax.scatter(combined["H1_count"], combined["pct_winners"],
               s=200, color="coral", edgecolor="black", zorder=5)
    for idx, row in combined.iterrows():
        ax.annotate(idx, (row["H1_count"], row["pct_winners"]),
                    textcoords="offset points", xytext=(10, 5), fontsize=9)
    ax.set_xlabel("H1 Count (# Loops)\n(Undeveloped Voids)")
    ax.set_ylabel("% Homeowners Paying Less")
    ax.set_title("Voids vs Tax Shift Benefit")
    ax.axhline(50, color="red", linestyle="--", alpha=0.5)
    
    ax = axes[2]
    ax.scatter(combined["H0_entropy"], combined["median_site_ratio"],
               s=200, color="green", edgecolor="black", zorder=5)
    for idx, row in combined.iterrows():
        ax.annotate(idx, (row["H0_entropy"], row["median_site_ratio"]),
                    textcoords="offset points", xytext=(10, 5), fontsize=9)
    ax.set_xlabel("H0 Entropy\n(Development Uniformity)")
    ax.set_ylabel("Median Site Value Ratio")
    ax.set_title("Uniformity vs Land Ratio")
    
    plt.suptitle("Linking Urban Topology to Tax Shift Outcomes",
                 fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig("../outputs/figures/07_topo_vs_fiscal.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Need 2+ metros with topology data for comparison plots.")
    print(f"Currently have: {len(combined)} metros with topology data.")

### 11. Persistence Barcode Visualization

In [ ]:
# Barcode plots give a different perspective than persistence diagrams
fig, axes = plt.subplots(n_metros, 2, figsize=(14, 4 * n_metros))

if n_metros == 1:
    axes = axes.reshape(1, 2)

for i, (metro, diagrams) in enumerate(topo_diagrams.items()):
    for dim in range(2):
        ax = axes[i, dim]
        dgm = diagrams[dim]
        finite = dgm[np.isfinite(dgm[:, 1])]
        
        if len(finite) == 0:
            ax.text(0.5, 0.5, "No features", ha="center", va="center",
                    transform=ax.transAxes)
            continue
        
        # Sort by birth time
        sorted_dgm = finite[finite[:, 0].argsort()]
        
        # Plot barcodes (horizontal lines from birth to death)
        for j, (birth, death) in enumerate(sorted_dgm):
            color = "steelblue" if dim == 0 else "coral"
            ax.plot([birth, death], [j, j], color=color, linewidth=1.5, alpha=0.7)
        
        ax.set_xlabel("Filtration Value (km)")
        ax.set_ylabel("Feature Index")
        ax.set_title(f"{metro} \u2014 H{dim} Barcode")

plt.suptitle("Persistence Barcodes by Metro",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../outputs/figures/07_barcodes.png", dpi=150, bbox_inches="tight")
plt.show()

### 12. Wasserstein Distance Between Metro Persistence Diagrams

In [ ]:
# If we have 2+ metros, compute pairwise Wasserstein distances
# between their persistence diagrams to quantify how different
# their urban structures are

if len(topo_diagrams) >= 2:
    metros_list = list(topo_diagrams.keys())
    
    for dim in range(2):
        print(f"\n=== H{dim} Wasserstein Distances ===")
        
        n = len(metros_list)
        dist_matrix = np.zeros((n, n))
        
        for i in range(n):
            for j in range(i + 1, n):
                dgm_i = topo_diagrams[metros_list[i]][dim]
                dgm_j = topo_diagrams[metros_list[j]][dim]
                
                # Filter to finite
                dgm_i = dgm_i[np.isfinite(dgm_i[:, 1])]
                dgm_j = dgm_j[np.isfinite(dgm_j[:, 1])]
                
                if len(dgm_i) > 0 and len(dgm_j) > 0:
                    dist = persim.wasserstein(dgm_i, dgm_j)
                    dist_matrix[i, j] = dist
                    dist_matrix[j, i] = dist
        
        dist_df = pd.DataFrame(dist_matrix, 
                                index=metros_list, columns=metros_list)
        print(dist_df.to_string())
        
        # Heatmap
        if n >= 2:
            fig, ax = plt.subplots(figsize=(8, 6))
            sns.heatmap(dist_df, annot=True, fmt=".3f", cmap="YlOrRd",
                        ax=ax, square=True)
            ax.set_title(f"H{dim} Wasserstein Distance Between Metros")
            plt.tight_layout()
            plt.savefig(f"../outputs/figures/07_wasserstein_H{dim}.png",
                        dpi=150, bbox_inches="tight")
            plt.show()
else:
    print("Need 2+ metros for Wasserstein distance computation.")

## 13. Interpretation Summary

**H0 (Connected Components) \u2014 Development Fragmentation:**
- Higher H0 count and persistence \u2192 more fragmented urban fabric
  with isolated pockets separated by undeveloped land
- A land-value tax would increase holding costs for these gaps,
  incentivizing infill development

**H1 (Loops) \u2014 Development Voids:**
- H1 features indicate "rings" of development encircling undeveloped
  or underutilized parcels
- These are prime targets for the proposed tax shift \u2014 high-value
  locations held out of productive use

**Wasserstein Distances \u2014 Structural Similarity:**
- Metros with similar topological fingerprints may respond similarly
  to the proposed tax shift
- This provides a principled basis for transferring policy predictions
  between comparable cities

## 14. Save All Results

In [ ]:
# Save fiscal comparison
comparison.to_csv("../outputs/reports/cross_metro_comparison.csv", index=False)

# Save topological features
topo_df.to_csv("../outputs/reports/topological_features.csv")

# Save combined analysis
if len(combined) > 0:
    combined.to_csv("../outputs/reports/combined_fiscal_topo.csv")

print("Saved: ../outputs/reports/cross_metro_comparison.csv")
print("Saved: ../outputs/reports/topological_features.csv")
print("Saved: ../outputs/reports/combined_fiscal_topo.csv")